# Vérifications du remplacement ×10
## tl;dr
Le nouveau meilleur checkpoint donne 9 910/10 000 sur MNIST. Les 12 170 entrées Cascade sont des copies exactes, réservées au train.
## Context & Methods
Compagnon du compte rendu, exécuté cellule par cellule avec Python. Exécuter depuis la racine du dépôt avec l’environnement `.venv`. Les contrôles ne réentraînent aucun modèle.
### Key Assumptions
Les annotations terrain historiques sont conservées. Les tests CTN ne sont pas indépendants. Le nouveau ×10 est entraîné avec une seule graine.
## Data
Les CSV MNIST, CTN, terrain et le manifeste du dataset sont les sources des chiffres.

### Vérifier les copies exactes


In [1]:
from pathlib import Path
import sys, json, csv, hashlib
import numpy as np
ROOT = Path.cwd()
assert (ROOT / 'scripts').is_dir(), 'Exécuter depuis la racine du dépôt.'
sys.path.insert(0, str(ROOT))
from scripts import _bootstrap
from reconnaissance_chiffres.entrainement_cascade import repeter_cascade, empreintes_images
p = ROOT / 'donnees/preparees/cascade/cascade_top_n_v1'
x = np.load(p / 'dataset_numpy/x_train_cascade.npy')
y = np.load(p / 'dataset_numpy/y_train_cascade.npy')
images, labels = repeter_cascade(x, y, 10)
assert images.shape == (12170,28,28,1) and labels.shape == (12170,)
print({'originaux': len(x), 'copies':len(images)-len(x), 'classes':np.bincount(y).tolist()})

{'originaux': 1217, 'copies': 10953, 'classes': [98, 100, 228, 102, 99, 103, 187, 100, 100, 100]}


### Vérifier les partitions


In [2]:
with np.load(ROOT / '.cache/keras/datasets/mnist.npz') as mnist:
    test = mnist['x_test'].astype('float32')[...,None] / 255
    train_mnist = mnist['x_train'].astype('float32')[...,None] / 255
a = np.load(ROOT / 'modeles/actifs/best_relu_10xcascade/indices_split.npz')
assert not set(a['mnist_train']) & set(a['mnist_validation'])
assert not empreintes_images(x) & empreintes_images(test)
assert not empreintes_images(x) & empreintes_images(train_mnist[a['mnist_validation']])
print({'mnist_train':len(a['mnist_train']),'mnist_validation':len(a['mnist_validation']),'cascade_validation_test':0})

{'mnist_train': 51000, 'mnist_validation': 9000, 'cascade_validation_test': 0}


## Results


In [3]:
from scripts.evaluation.bilan_remplacement_relu_10x import calculer
bilan = calculer()
print(json.dumps({'mnist':[{k:r[k] for k in ['modele','corrects','erreurs','accuracy']} for r in bilan['mnist']], 'cascade':[{k:r[k] for k in ['modele','top_1_corrects','top_5_corrects','evaluation_independante']} for r in bilan['cascade']], 'preservation':bilan['preservation']},ensure_ascii=False,indent=2))

{
  "mnist": [
    {
      "modele": "Ancien ×10 augmenté, meilleur checkpoint",
      "corrects": 9910,
      "erreurs": 90,
      "accuracy": 0.991
    },
    {
      "modele": "Nouveau ×10 exact, meilleur checkpoint",
      "corrects": 9910,
      "erreurs": 90,
      "accuracy": 0.991
    }
  ],
  "cascade": [
    {
      "modele": "Ancien ×10 augmenté",
      "top_1_corrects": 1209,
      "top_5_corrects": 1222,
      "evaluation_independante": false
    },
    {
      "modele": "Nouveau ×10 exact",
      "top_1_corrects": 1215,
      "top_5_corrects": 1221,
      "evaluation_independante": false
    }
  ],
  "preservation": {
    "autres_modeles_fichiers_identiques": 18,
    "autres_resultats_fichiers_identiques": 5693,
    "annotations_reference_identiques": 3,
    "comparatifs_autres_modeles_lignes_identiques": true,
    "terrain_pretraitements_verifies": 139,
    "ctn_zones_verifiees": 1227
  }
}


## Takeaways
Les copies sont exactes, les labels sont alignés et les partitions MNIST sont disjointes. Les résultats CTN utilisent les sources d’entraînement ; les photos non annotées n’ont pas de score d’exactitude.